# HPDM097 Assignment 2

## Iteration 4: Auditing process and probability of delay (Gemini)

This is the 4th iteration to the code. Here we start the auditing process, turn the model into and unconstrained model and measure the probability of delay.

Prompt and response: https://gemini.google.com/share/70d82ad6fefa

## Prompt:
Let's start writing the audit logic. First, produce a daily audit of the occupancy of the acute stroke unit and over time, construct the occupancy probability distribution function (PDF). As the model has no capacity limits, daily occupancy is Poisson distributed. The main outcome measure we are interested in the probability that a patient cannot be immediately admitted to the acute unit, rehab ward, or ESD.  Let's call this estimate the probability of delay or shorthand p(delay). In the future, we will construct different scenarios with a range of different bed numbers and estimate p(delay) to construct a stepped trade-off curve. Use both the PDF and cumulative probability density function of occupancy to calculate the probability of delay. The general form of this calculation, often referred to as the Erlang loss formula, is P(N= n)/P(N ≤ n).

## Response
This is a brilliant analytical step. Shifting from a constrained model to an unconstrained (infinite capacity) model to measure the "offered load" is the standard best practice for healthcare capacity planning.

You are entirely correct about the underlying mathematics. If we assume arrivals are a Poisson process (derived from our exponential inter-arrival times) and we have no capacity limits, the daily occupancy naturally forms a Poisson distribution. The probability of a patient facing a delay when the true capacity is n can be estimated using the Erlang loss formula (Erlang B), which expresses the blocking probability exactly as you described.
 
To achieve this, I have made the following iterative changes:

Infinite Capacity: Set the resource capacities to 9999 (SimPy's practical equivalent of infinity) so patients never queue, revealing the true unconstrained occupancy.

Daily Audit Process: Added a daily_audit generator that wakes up every 1.0 days to record len(self.acute_beds.users).

Extended Run Time: Increased the simulation time to 3,650 days (10 years) and turned TRACE off to ensure we generate a statistically stable and smooth empirical PDF.

Mathematical Calculation: Added a post-run calculation block to compute the empirical PDF, CDF, and finally P 
delay for our target of 10 acute beds.

Here is the updated code:

In [1]:
import simpy
import numpy as np

# ---------------------------------------------------------
# 2. Trace utility function as a debug print switch
# ---------------------------------------------------------
TRACE = False # Turned off for long simulation run

def trace(msg):
    if TRACE:
        print(msg)

# ---------------------------------------------------------
# 1. Build and sample from distribution classes
# ---------------------------------------------------------
class ExponentialDistribution:
    def __init__(self, mean, random_seed=None):
        self.mean = mean
        self.rng = np.random.default_rng(seed=random_seed)
        
    def sample(self):
        return self.rng.exponential(scale=self.mean)

class LognormalDistribution:
    def __init__(self, sample_mean, sample_stdev, random_seed=None):
        variance = sample_stdev ** 2
        self.mu = np.log((sample_mean ** 2) / np.sqrt(variance + sample_mean ** 2))
        self.sigma = np.sqrt(np.log(1 + (variance / sample_mean ** 2)))
        self.rng = np.random.default_rng(seed=random_seed)
        
    def sample(self):
        return self.rng.lognormal(mean=self.mu, sigma=self.sigma)

# ---------------------------------------------------------
# 3. Create a parameter container class called Scenario
# ---------------------------------------------------------
class Scenario:
    def __init__(self):
        self.random_seed = 42
        
        # Set to practically infinite to measure unconstrained offered load
        self.acute_beds_capacity = 9999 
        self.rehab_beds_capacity = 9999 
        
        # Target capacities for our p(delay) evaluation
        self.target_acute_beds = 10
        
        self.acute_inter_arrival_means = {
            'Acute stroke': 1.2, 'TIA': 9.3, 'Complex neurological': 3.6, 'Other': 3.2
        }
        
        self.rehab_inter_arrival_means = {
            'Acute stroke': 21.8, 'Complex neurological': 31.7, 'Other': 28.6
        }
        
        self.acute_destinations = ['Rehab', 'ESD', 'Other']
        self.acute_transfer_matrix = {
            'Acute stroke': [0.24, 0.13, 0.63], 'TIA': [0.01, 0.01, 0.98],
            'Complex neurological': [0.11, 0.05, 0.84], 'Other': [0.05, 0.10, 0.85]
        }
        
        self.rehab_destinations = ['ESD', 'Other']
        self.rehab_transfer_matrix = {
            'Acute stroke': [0.40, 0.60], 'TIA': [0.00, 1.00],
            'Complex neurological': [0.09, 0.91], 'Other': [0.13, 0.87] 
        }
        
        self.acute_los_params = {
            'Acute stroke': {'Rehab': (7.4, 8.6), 'ESD': (4.6, 4.8), 'Other': (7.4, 8.6)},
            'TIA': {dest: (1.8, 2.3) for dest in self.acute_destinations},
            'Complex neurological': {dest: (4.0, 5.0) for dest in self.acute_destinations},
            'Other': {dest: (3.8, 5.2) for dest in self.acute_destinations}
        }
        
        self.rehab_los_params = {
            'Acute stroke': {'ESD': (30.3, 23.1), 'Other': (28.4, 27.2)},
            'TIA': {dest: (18.7, 23.5) for dest in self.rehab_destinations},
            'Complex neurological': {dest: (27.6, 28.4) for dest in self.rehab_destinations},
            'Other': {dest: (16.1, 14.1) for dest in self.rehab_destinations}
        }

# ---------------------------------------------------------
# 4. Create a patient class called AcutePatient
# ---------------------------------------------------------
class AcutePatient:
    def __init__(self, p_id, p_type, arrival_time, source="Acute"):
        self.id = p_id
        self.type = p_type
        self.arrival_time = arrival_time
        self.source = source
        self.acute_destination = None
        self.acute_bed_request_time = None
        self.acute_bed_acquire_time = None
        self.acute_length_of_stay = None
        self.acute_discharge_time = None
        self.rehab_destination = None
        self.rehab_bed_request_time = None
        self.rehab_bed_acquire_time = None
        self.rehab_length_of_stay = None
        self.rehab_discharge_time = None

# ---------------------------------------------------------
# 5. Create a model class called StrokeUnit
# ---------------------------------------------------------
class StrokeUnit:
    def __init__(self, env, scenario):
        self.env = env
        self.scenario = scenario
        self.patient_counter = 0
        
        # New Audit Data Structure
        self.daily_acute_occupancy = []
        
        self.acute_beds = simpy.Resource(env, capacity=self.scenario.acute_beds_capacity)
        self.rehab_beds = simpy.Resource(env, capacity=self.scenario.rehab_beds_capacity)
        
        self.acute_routing_rng = np.random.default_rng(seed=self.scenario.random_seed + 100)
        self.rehab_routing_rng = np.random.default_rng(seed=self.scenario.random_seed + 200)
        
        self.acute_arrival_dists = {}
        self.rehab_arrival_dists = {}
        self.acute_los_dists = {p_type: {} for p_type in self.scenario.acute_inter_arrival_means.keys()}
        self.rehab_los_dists = {p_type: {} for p_type in self.scenario.acute_inter_arrival_means.keys()}
        
        dist_seed = self.scenario.random_seed + 1000
        
        for p_type, mean_iat in self.scenario.acute_inter_arrival_means.items():
            self.acute_arrival_dists[p_type] = ExponentialDistribution(mean_iat, dist_seed)
            dist_seed += 1
            for dest in self.scenario.acute_destinations:
                mean, stdev = self.scenario.acute_los_params[p_type][dest]
                self.acute_los_dists[p_type][dest] = LognormalDistribution(mean, stdev, dist_seed)
                dist_seed += 1
                
        for p_type, mean_iat in self.scenario.rehab_inter_arrival_means.items():
            self.rehab_arrival_dists[p_type] = ExponentialDistribution(mean_iat, dist_seed)
            dist_seed += 1
            
        for p_type in self.scenario.acute_inter_arrival_means.keys():
            for dest in self.scenario.rehab_destinations:
                mean, stdev = self.scenario.rehab_los_params[p_type][dest]
                self.rehab_los_dists[p_type][dest] = LognormalDistribution(mean, stdev, dist_seed)
                dist_seed += 1

    def daily_audit(self):
        """Wakes up every 1 day to record the number of patients currently in an acute bed."""
        while True:
            # len(self.acute_beds.users) gives the active, unconstrained occupancy
            self.daily_acute_occupancy.append(len(self.acute_beds.users))
            yield self.env.timeout(1.0)

    def acute_arrivals_generator(self, patient_type):
        while True:
            yield self.env.timeout(self.acute_arrival_dists[patient_type].sample())
            self.patient_counter += 1
            patient = AcutePatient(self.patient_counter, patient_type, self.env.now, "Acute")
            trace(f"Day {self.env.now:06.2f} | P{patient.id:04d} ({patient.type}) arrived at ACUTE.")
            self.env.process(self.acute_process(patient))

    def rehab_direct_arrivals_generator(self, patient_type):
        while True:
            yield self.env.timeout(self.rehab_arrival_dists[patient_type].sample())
            self.patient_counter += 1
            patient = AcutePatient(self.patient_counter, patient_type, self.env.now, "Direct Rehab")
            trace(f"Day {self.env.now:06.2f} | P{patient.id:04d} ({patient.type}) arrived at REHAB.")
            self.env.process(self.rehab_process(patient))

    def acute_process(self, patient):
        probs = self.scenario.acute_transfer_matrix[patient.type]
        patient.acute_destination = self.acute_routing_rng.choice(self.scenario.acute_destinations, p=probs)
        patient.acute_bed_request_time = self.env.now
        
        with self.acute_beds.request() as req:
            yield req
            patient.acute_bed_acquire_time = self.env.now
            los = self.acute_los_dists[patient.type][patient.acute_destination].sample()
            patient.acute_length_of_stay = los
            yield self.env.timeout(los)
            patient.acute_discharge_time = self.env.now
            trace(f"Day {self.env.now:06.2f} | P{patient.id:04d} ({patient.type}) left ACUTE.")
            
            if patient.acute_destination == 'Rehab':
                self.env.process(self.rehab_process(patient))

    def rehab_process(self, patient):
        probs = self.scenario.rehab_transfer_matrix[patient.type]
        patient.rehab_destination = self.rehab_routing_rng.choice(self.scenario.rehab_destinations, p=probs)
        patient.rehab_bed_request_time = self.env.now
        
        with self.rehab_beds.request() as req:
            yield req
            patient.rehab_bed_acquire_time = self.env.now
            los = self.rehab_los_dists[patient.type][patient.rehab_destination].sample()
            patient.rehab_length_of_stay = los
            yield self.env.timeout(los)
            patient.rehab_discharge_time = self.env.now
            trace(f"Day {self.env.now:06.2f} | P{patient.id:04d} ({patient.type}) left REHAB.")

    def run_simulation(self):
        # Start the audit process
        self.env.process(self.daily_audit())
        
        for p_type in self.scenario.acute_inter_arrival_means.keys():
            self.env.process(self.acute_arrivals_generator(p_type))
            
        for p_type in self.scenario.rehab_inter_arrival_means.keys():
            self.env.process(self.rehab_direct_arrivals_generator(p_type))

# =========================================================
# Execution & Mathematical Audit Logic
# =========================================================
if __name__ == "__main__":
    print("--- Running 10-Year Simulation to generate Occupancy PDF ---")
    scenario = Scenario()
    env = simpy.Environment()
    model = StrokeUnit(env, scenario)
    model.run_simulation()
    
    # Run for 10 years (3650 days) to get stable empirical distribution
    env.run(until=3650)
    
    # Process Audit Data
    occupancy_data = model.daily_acute_occupancy
    n = scenario.target_acute_beds
    
    # Calculate empirical PDF
    max_occ = max(max(occupancy_data), n) # Ensure array goes up to at least n
    counts = np.zeros(max_occ + 1)
    for occ in occupancy_data:
        counts[occ] += 1
        
    pdf = counts / len(occupancy_data)
    
    # Calculate empirical CDF
    cdf = np.cumsum(pdf)
    
    # Calculate p(delay) using Erlang logic: P(N=n) / P(N<=n)
    p_n = pdf[n]
    p_le_n = cdf[n]
    
    p_delay = p_n / p_le_n if p_le_n > 0 else 0.0

    print(f"\n--- Audit Results (Target Acute Capacity: {n}) ---")
    print(f"Mean Unconstrained Occupancy: {np.mean(occupancy_data):.2f} beds")
    print(f"P(N = {n}):  {p_n:.4f}  (PDF at target)")
    print(f"P(N <= {n}): {p_le_n:.4f}  (CDF at target)")
    print(f"Probability of Delay (Erlang estimate): {p_delay:.4f}  ({p_delay*100:.2f}%)")

--- Running 10-Year Simulation to generate Occupancy PDF ---

--- Audit Results (Target Acute Capacity: 10) ---
Mean Unconstrained Occupancy: 8.09 beds
P(N = 10):  0.1055  (PDF at target)
P(N <= 10): 0.8074  (CDF at target)
Probability of Delay (Erlang estimate): 0.1306  (13.06%)
